# 🏨 Hotel do Mosquito — Acesso via Navegador

**Laboratório de Banco de Dados**  
Camili de Moura Marangoni · Lucas Sobrinho Santos · Maria Eduarda Patu Ângelo da Silva  
Matheus Pinheiro de Camargo Silva · Willian Alexandre Schwingel Ferreira

---

## Como usar

Execute as **3 células abaixo em ordem**. Clique em ▶ e **aguarde o ✅** antes de passar para a próxima.

| Célula | O que faz | Tempo |
|---|---|---|
| **Célula 1** | Instala MySQL, display virtual, openbox, noVNC, cloudflared | ~2 min |
| **Célula 2** | Carrega o banco de dados | ~15 s |
| **Célula 3** | Abre o sistema e exibe o link | ~30 s |

### Credenciais
| Login | Senha | Perfil |
|---|---|---|
| gerente1 | senha123 | Gerente |
| recep1 | senha123 | Recepcionista |

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CÉLULA 1 — Instalar ambiente (~2 minutos)
# ══════════════════════════════════════════════════════════════════
import os, sys, subprocess, configparser, time
from pathlib import Path

def sh(cmd): return subprocess.run(cmd, shell=True, capture_output=True, text=True)
def ok(msg): print(f'  ✅ {msg}')
def step(msg, cmd):
    print(f'  ⏳ {msg}...')
    sh(cmd)

print('📦 PASSO 1/6 — Pacotes do sistema...')
step('MySQL Server',       'apt-get install -y mysql-server > /dev/null 2>&1')
step('Xvfb + x11vnc',      'apt-get install -y xvfb x11vnc websockify xauth > /dev/null 2>&1')
step('openbox (WM)',        'apt-get install -y openbox > /dev/null 2>&1')
step('x11-utils',           'apt-get install -y x11-utils > /dev/null 2>&1')
ok('Pacotes de sistema instalados')

print('\n📦 PASSO 2/6 — noVNC...')
step('noVNC', 'git clone --depth=1 https://github.com/novnc/noVNC.git /opt/noVNC > /dev/null 2>&1 || true')
ok('noVNC pronto')

print('\n📦 PASSO 3/6 — cloudflared...')
step('Download', 'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared')
ok('cloudflared pronto')

print('\n🔐 PASSO 4/6 — MySQL...')
sh('service mysql start')
time.sleep(3)
sh("mysql -e \"ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY 'hotel123'; FLUSH PRIVILEGES;\"")
ok('MySQL pronto')

print('\n📂 PASSO 5/6 — Repositório...')
if not Path('hotel-mosquito-labdb').exists():
    step('Clonando', 'git clone https://github.com/Matheus-PC-Silva/hotel-mosquito-labdb.git -q')
else:
    step('Atualizando', 'git -C hotel-mosquito-labdb pull -q')
ok('Repositório pronto')

print('\n📦 PASSO 6/6 — Python...')
step('Pacotes', f'{sys.executable} -m pip install mysql-connector-python==8.4.0 tkcalendar==1.6.1 -q')

cfg_path = Path('hotel-mosquito-labdb/app/config.ini')
cfg = configparser.ConfigParser()
cfg.read(cfg_path)
cfg['database']['port']     = '3306'
cfg['database']['host']     = 'localhost'
cfg['database']['password'] = 'hotel123'
with open(cfg_path, 'w') as f:
    cfg.write(f)

proj = str(Path('hotel-mosquito-labdb').resolve())
if proj not in sys.path:
    sys.path.insert(0, proj)
ok('Pacotes Python + config.ini prontos')

print('\n✅ AMBIENTE PRONTO — Execute a Célula 2')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CÉLULA 2 — Carregar banco de dados (~15 segundos)
# ══════════════════════════════════════════════════════════════════
import subprocess
from pathlib import Path

print('🗄️  Carregando schema, procedures, triggers e dados...')
result = subprocess.run(
    ['mysql', '-uroot', '-photel123', '--default-character-set=utf8mb4'],
    input=Path('hotel-mosquito-labdb/sql/hotel_mosquito_full.sql').read_bytes(),
    capture_output=True
)
if result.returncode != 0:
    print('❌ Erro:', result.stderr.decode(errors='replace'))
    raise SystemExit('Verifique o erro acima.')

check = subprocess.run(
    ['mysql', '-uroot', '-photel123', 'hotel_mosquito', '-t', '-e',
     'SELECT (SELECT COUNT(*) FROM Categoria_Quarto) categorias,'
     '(SELECT COUNT(*) FROM Quarto) quartos,'
     '(SELECT COUNT(*) FROM Cliente) clientes,'
     '(SELECT COUNT(*) FROM Funcionario) funcionarios,'
     '(SELECT COUNT(*) FROM Reserva) reservas,'
     '(SELECT COUNT(*) FROM Hospedagem) hospedagens;'],
    capture_output=True, text=True
)
print('\n📊 Dados (esperado: 5 | 15 | 10 | 5 | 10 | 8):')
print(check.stdout)
print('✅ BANCO PRONTO — Execute a Célula 3')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CÉLULA 3 — Abrir o sistema no navegador (~30 segundos)
# ══════════════════════════════════════════════════════════════════
import subprocess, os, re, sys, time
from pathlib import Path

proj_dir  = str(Path('hotel-mosquito-labdb').resolve())
LOG       = '/tmp/hotel_app.log'
env99     = {**os.environ, 'DISPLAY': ':99'}
DEVNULL   = subprocess.DEVNULL

# ── 1. Display virtual ──────────────────────────────────────────
print('🖥️  Display virtual...')
subprocess.Popen(
    ['Xvfb', ':99', '-screen', '0', '1280x800x24', '-ac', '+extension', 'GLX'],
    stdout=DEVNULL, stderr=DEVNULL
)
time.sleep(2)
os.environ['DISPLAY'] = ':99'

# Fundo cinza escuro para confirmar que o display está ativo
subprocess.run(['xsetroot', '-solid', '#2d3748'], env=env99,
               stdout=DEVNULL, stderr=DEVNULL)
print('  ✅ Display :99 ativo')

# ── 2. Gerenciador de janelas ────────────────────────────────────
print('🪟  Gerenciador de janelas (openbox)...')
subprocess.Popen(['openbox', '--sm-disable'],
                 env=env99, stdout=DEVNULL, stderr=DEVNULL)
time.sleep(2)
print('  ✅ openbox iniciado')

# ── 3. VNC server ───────────────────────────────────────────────
print('📡 VNC server...')
subprocess.Popen(
    ['x11vnc', '-display', ':99', '-forever', '-nopw', '-quiet', '-xkb'],
    stdout=DEVNULL, stderr=DEVNULL
)
time.sleep(2)
print('  ✅ VNC pronto (porta 5900)')

# ── 4. noVNC proxy ──────────────────────────────────────────────
print('🌐 noVNC proxy...')
subprocess.Popen(
    ['python3', '-m', 'websockify', '--web=/opt/noVNC', '6080', 'localhost:5900'],
    stdout=DEVNULL, stderr=DEVNULL
)
time.sleep(2)
print('  ✅ noVNC pronto (porta 6080)')

# ── 5. Túnel Cloudflare ──────────────────────────────────────────
print('🔗 Criando túnel Cloudflare (sem login)...')
cf = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:6080'],
    stdout=DEVNULL, stderr=subprocess.PIPE, text=True
)
url = None
for _ in range(50):
    line = cf.stderr.readline()
    m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
if not url:
    print('  ⚠️  Túnel não retornou URL — re-execute esta célula')
    url = 'http://localhost:6080'
else:
    print(f'  ✅ Túnel ativo')

# ── 6. Aplicação ────────────────────────────────────────────────
print('🚀 Iniciando Hotel do Mosquito...')
with open(LOG, 'w') as log:
    app = subprocess.Popen(
        [sys.executable, '-m', 'app.main'],
        cwd=proj_dir, env=env99,
        stdout=log, stderr=log
    )
time.sleep(5)

if app.poll() is not None:          # app encerrou antes do esperado
    print('\n❌ O sistema encerrou com erro. Log:')
    print(Path(LOG).read_text(errors='replace') or '(sem saída)')
    raise SystemExit('Corrija o erro acima e execute esta célula novamente.')

# ── 7. Link final ───────────────────────────────────────────────
link = f"{url}/vnc.html?autoconnect=true&resize=scale&quality=8"
print()
print('═' * 62)
print('  🎉 SISTEMA PRONTO!')
print('═' * 62)
print(f'\n  👉  {link}\n')
print('  Credenciais:')
print('  gerente1 / senha123  →  Gerente')
print('  recep1   / senha123  →  Recepcionista')
print()
print('  ⚠️  Mantenha esta aba aberta — fechar encerra o sistema.')
print('═' * 62)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# DIAGNÓSTICO (execute só se aparecer tela preta ou erro)
# ══════════════════════════════════════════════════════════════════
import subprocess
from pathlib import Path

print('=== Log do app ===')
log = Path('/tmp/hotel_app.log').read_text(errors='replace')
print(log if log.strip() else '(vazio — o app pode ainda estar iniciando)')

print('\n=== MySQL rodando? ===')
r = subprocess.run(['service', 'mysql', 'status'], capture_output=True, text=True)
print('Ativo' if 'active' in r.stdout.lower() or 'running' in r.stdout.lower() else r.stdout[:300])

print('\n=== Processos no display :99 ===')
r = subprocess.run(['bash', '-c', 'DISPLAY=:99 xdpyinfo 2>&1 | head -5'], capture_output=True, text=True)
print(r.stdout or r.stderr[:300])

print('\n=== VNC porta 5900 ===')
r = subprocess.run(['bash', '-c', 'ss -tlnp | grep 5900'], capture_output=True, text=True)
print(r.stdout or '(porta não encontrada)')

print('\n=== noVNC porta 6080 ===')
r = subprocess.run(['bash', '-c', 'ss -tlnp | grep 6080'], capture_output=True, text=True)
print(r.stdout or '(porta não encontrada)')

---
## ℹ️ Informações adicionais

### Tela preta ou link não apareceu?
Execute a **célula de diagnóstico** acima — ela mostra exatamente onde está o problema.

### O Colab desconectou?
Execute as **Células 1, 2 e 3** em ordem novamente.

### Dicas no celular
- **Toque simples** → clique do mouse  
- **Toque longo** → clique direito  
- **Teclado** → aparece ao tocar em campos de texto  
- **Pinça** → zoom extra na tela

---
*Hotel do Mosquito — LABDB · https://github.com/Matheus-PC-Silva/hotel-mosquito-labdb*